# Bài 19: Cách mọi thứ kết nối

Bạn đã sử dụng sản phẩm để tạo bài viết. Nhưng điều gì xảy ra **đằng sau** khi bạn nói "Tạo bài viết về SEO"?

Bài học này theo dõi **toàn bộ hành trình của một yêu cầu** — từ lúc bạn gõ tin nhắn đến lúc bài viết xuất hiện trên ổ đĩa. Không cần gọi API — chỉ là một chuyến tham quan cách các phần kết nối với nhau.

## Hành trình của một yêu cầu

Khi bạn gõ "Tạo bài viết về on-page SEO" trong giao diện web:

```
Bạn (trình duyệt)
  |
  v
serve.py --- AgentOS nhận yêu cầu qua SSE
  |
  v
agents/team.py --- Team Leader (Sonnet) quyết định giao cho ai
  |
  v
agents/content_writer.py --- Content Writer thực hiện:
  |                            1. web_search() qua DataForSEO
  |                            2. Viết bài viết
  |                            3. save_article() lưu xuống ổ đĩa
  |
  v
tools/storage.py --- Lưu content/{id}.md + cập nhật articles.json
  |
  v
content/ --- Bài viết được lưu dưới dạng file .md
```

Mỗi file có **một trách nhiệm duy nhất**. Hãy đi qua từng file một:

## serve.py — Web backend

`serve.py` là **điểm vào web** — file bạn chạy với `python output/backend/serve.py`.

Nó làm ba việc:
1. Kiểm tra `ANTHROPIC_API_KEY`
2. Bọc team bằng **AgentOS** (tự động tạo hơn 50 API endpoint)
3. Thêm các route (đường dẫn) tùy chỉnh cho bài viết (`/api/articles`)

AgentOS tự động tạo các endpoint như:
- `POST /teams/seo-workspace/runs` — gửi prompt (hỗ trợ SSE streaming)
- `GET /health` — kiểm tra tình trạng
- `GET /docs` — tài liệu Swagger API

Các route tùy chỉnh cung cấp tầng storage:
- `GET /api/articles` — liệt kê tất cả bài viết
- `GET /api/articles/{id}` — xem một bài viết
- `DELETE /api/articles/{id}` — xóa một bài viết

`chat.py` cũng tồn tại như phương án CLI thay thế — cùng team, khác giao diện.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

# Let's look at serve.py
serve_path = os.path.abspath("../../output/backend/serve.py")
with open(serve_path, "r", encoding="utf-8") as f:
    serve_code = f.read()

print(f"serve.py ({len(serve_code.splitlines())} lines)\n")
for i, line in enumerate(serve_code.splitlines(), 1):
    print(f"  {i:3d} | {line}")

## agents/team.py — Team

Định nghĩa team bao gồm:
- **Team Leader** (Claude Sonnet) — đọc yêu cầu của bạn và ủy thác
- **3 thành viên**, mỗi người là agent Sonnet với tools riêng:
  - **Content Writer** — nghiên cứu chủ đề và viết bài (DataForSEO search + storage)
  - **Image Finder** — tìm và chèn hình ảnh vào bài viết (tùy chọn, DataForSEO Images + storage)
  - **AIO Analyzer** — phân tích Google AI Overviews (AIO tools)

Leader không bao giờ gọi tools trực tiếp — chỉ ủy thác cho đúng thành viên.

Đây là mô hình **quản lý dự án**: bạn nói chuyện với một người, họ phân việc cho ai phù hợp.

In [ ]:
# Let's look at the team structure
team_path = os.path.abspath("../../output/backend/agents/team.py")
with open(team_path, "r", encoding="utf-8") as f:
    team_code = f.read()

print(f"agents/team.py ({len(team_code.splitlines())} lines)\n")
for i, line in enumerate(team_code.splitlines(), 1):
    print(f"  {i:3d} | {line}")

## agents/ — Các AI worker

Thư mục `agents/` chứa mỗi agent trong một file riêng:

| File | Agent | Tools | Mục đích |
|------|-------|-------|----------|
| `content_writer.py` | Content Writer | DataForSEO search + save_article + list_all_articles | Nghiên cứu và viết bài SEO |
| `image_finder.py` | Image Finder | DataForSEO Images + get_article_content + update_article_content | Tìm và chèn hình ảnh |
| `aio_analyzer.py` | AIO Analyzer | analyze_keyword_aio + optimize_for_aio | Phân tích Google AI Overviews |
| `team.py` | Tập hợp Team | (ủy thác) | Sonnet leader + 3 thành viên |
| `__init__.py` | — | — | Re-export mọi thứ |

**Tất cả agent đều dùng Claude Sonnet.** Leader và tất cả thành viên dùng cùng model.

**Một agent = một file.** Muốn thay đổi writer? Mở `content_writer.py`. Muốn thêm agent kiểm tra sự thật? Tạo `fact_checker.py`.

In [ ]:
# Let's look at the Content Writer definition
writer_path = os.path.abspath("../../output/backend/agents/content_writer.py")
with open(writer_path, "r", encoding="utf-8") as f:
    print(f.read())

## tools/ — Khả năng

Thư mục `tools/` chứa những gì agent **có thể làm**:

| File | Mục đích |
|------|----------|
| `storage.py` | Lưu trữ file cục bộ — lưu, liệt kê, đọc, cập nhật bài viết |
| `search.py` | Tìm kiếm web qua DataForSEO SERP API |
| `aio.py` | Phân tích AI Overview qua DataForSEO + hàm giải mã credentials |
| `images.py` | Tìm kiếm hình ảnh qua DataForSEO |
| `__init__.py` | Package marker |

`agents/` = **ai** (các AI worker với instructions riêng)

`tools/` = **gì** (các khả năng mà worker sử dụng)

Tất cả tools đều **fallback an toàn**:
- Không có DataForSEO key → search/images/AIO trả về kết quả rỗng, nhưng agent vẫn hoạt động
- `get_dataforseo_credentials()` trong `aio.py` là hàm giải mã credentials dùng chung

## Theo dõi luồng hình ảnh

Khi bạn nói "Thêm hình ảnh vào bài viết on-page-seo":

```
Yêu cầu của bạn
  |
  v
Team Leader --- quyết định đây là việc hình ảnh
  |
  v
Image Finder:
  1. get_article_content("on-page-seo")  --- đọc bài viết từ storage
  2. search_images("on page SEO")        --- tìm hình ảnh qua DataForSEO
  3. Chèn ![alt](url) vào Markdown
  4. update_article_content("on-page-seo", updated_markdown) --- lưu thay đổi
  |
  v
content/on-page-seo.md --- giờ đã có hình ảnh!
```

Chú ý cách Image Finder dùng **storage tools** (get + update) cộng với **image search tools**. Nó đọc bài viết có sẵn, thêm hình ảnh, rồi lưu phiên bản đã cập nhật.

## Tại sao cấu trúc file như vậy?

Tại sao không bỏ hết vào một file? Đây là triết lý:

**Một file = một việc.** `content_writer.py` viết bài. `storage.py` lưu file. `search.py` tìm kiếm web. Bạn không bao giờ phải đoán thứ gì nằm ở đâu.

**Một agent = một file.** Muốn thay đổi instructions của writer? Mở `agents/content_writer.py`. Muốn thêm agent kiểm tra sự thật? Tạo `agents/fact_checker.py` rồi thêm vào team.

**Đặt tên = điều hướng.** Tên file cho bạn biết bên trong có gì mà không cần mở ra. Ai đó nói "bài viết không lưu được", bạn nhìn vào `tools/storage.py`.

**`agents/` vs `tools/`** — Định nghĩa agent AI vs khả năng. Agent là tầng "suy nghĩ" (dùng LLM). Tools là tầng "thực thi" (gọi API, đọc file, lưu dữ liệu).

**Phẳng tốt hơn lồng nhau.** 3 thành viên làm việc trực tiếp. Không có pipeline.py điều phối các bước, không có workspace.py làm cầu nối. Mỗi thành viên có tools riêng và gọi trực tiếp.

## Bản đồ file hoàn chỉnh

```
output/
├── backend/                    (12 file Python)
│   ├── chat.py                 Điểm vào CLI (~40 dòng)
│   ├── serve.py                Web backend (AgentOS + route bài viết)
│   ├── agents/                 Định nghĩa agent (ai)
│   │   ├── __init__.py         Re-export mọi thứ
│   │   ├── content_writer.py   Content Writer (search + storage)
│   │   ├── image_finder.py     Image Finder (images + storage, tùy chọn)
│   │   ├── aio_analyzer.py     AIO Analyzer (AIO tools)
│   │   └── team.py             Agno Team (leader + 3 thành viên)
│   └── tools/                  Định nghĩa tools (gì)
│       ├── __init__.py         Package marker
│       ├── storage.py          Lưu trữ file cục bộ (JSON + .md)
│       ├── search.py           DataForSEO web search
│       ├── aio.py              Phân tích AIO + credentials
│       └── images.py           DataForSEO image search
└── frontend/                   React + Vite web app
    ├── package.json            Dependencies
    ├── vite.config.js          Proxy → backend
    └── src/                    Components + API client
```

In [ ]:
# Verify the file structure matches
output_dir = os.path.abspath("../../output/backend")
print("Your production codebase:\n")

for root, dirs, files in os.walk(output_dir):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(output_dir, "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            with open(filepath, "r", encoding="utf-8") as f:
                lines = len(f.readlines())
            print(f"{sub_indent}{file} ({lines} lines)")

## Tổng kết

Mọi thứ kết nối qua **lời gọi hàm Python đơn giản**. Không có phép thuật.

- **`serve.py`** — Web backend. AgentOS bọc team + route bài viết tùy chỉnh.
- **`chat.py`** — Phương án CLI thay thế. Cùng team, giao diện terminal.
- **`agents/team.py`** — Team Leader (Sonnet) ủy thác cho 3 thành viên Sonnet.
- **`agents/*.py`** — Một file mỗi agent. Mỗi agent có model, tools và instructions riêng.
- **`tools/storage.py`** — Lưu trữ file cục bộ. Bài viết dưới dạng `.md` + metadata JSON.
- **`tools/search.py`** — Tìm kiếm web qua DataForSEO.
- **`tools/aio.py`** — Phân tích AI Overview + credentials dùng chung.
- **`tools/images.py`** — Tìm kiếm hình ảnh qua DataForSEO.

Toàn bộ luồng: **serve.py → agents/team.py → agents/{member}.py → tools/{capability}.py → content/**

Triết lý: **một file = một việc, một agent = một file, đặt tên = điều hướng.**

**Bài tiếp theo:** Kiến thức web cơ bản — hiểu frontend, backend, API, và SSE trước khi xem giao diện web.

## Bài tập

Câu hỏi chỉ đọc — không cần gọi API. Bạn đang đọc code, giống như cách Claude Code khám phá codebase.

1. Mở `output/backend/agents/content_writer.py`. Content Writer có những tools nào? Chuyện gì xảy ra nếu `DATA_FOR_SEO_API_KEY` không được thiết lập?
2. Mở `output/backend/agents/image_finder.py`. Hàm `build_image_finder()` trả về gì nếu DataForSEO chưa được cấu hình?
3. Vẽ chuỗi import: `serve.py` import từ `agents/team.py`, rồi import từ `agents/content_writer.py`, rồi import từ `tools/search.py` và `tools/storage.py`. Vẽ sơ đồ import này ra giấy.
4. Tại sao `get_dataforseo_credentials()` nằm trong `tools/aio.py` mà không phải `tools/search.py`? (Gợi ý: ai khác cũng dùng nó?)

<details>
<summary>Bấm để xem đáp án</summary>

1. **Tools của Content Writer:** `DataForSEOSearchTools` (có điều kiện) + `save_article` + `list_all_articles`. Nếu không có DataForSEO key, search tool đơn giản không được thêm vào — writer vẫn hoạt động nhưng dựa vào kiến thức sẵn có.

2. **`build_image_finder()`** trả về `None` nếu DataForSEO chưa được cấu hình. Team kiểm tra `if image_finder is not None` trước khi thêm vào danh sách thành viên.

3. **Sơ đồ import:**
```
serve.py
  └── agents/team.py
        ├── agents/content_writer.py
        │     ├── tools/search.py
        │     ├── tools/storage.py
        │     └── tools/aio.py (credentials)
        ├── agents/image_finder.py
        │     ├── tools/images.py
        │     ├── tools/storage.py
        │     └── tools/aio.py (credentials)
        └── agents/aio_analyzer.py
              └── tools/aio.py
```

4. **`get_dataforseo_credentials()`** nằm trong `aio.py` vì nó được dùng chung bởi `search.py`, `images.py` và `aio.py` — cả ba đều dùng DataForSEO. Đặt nó trong `aio.py` tránh được import vòng tròn (circular imports) vì AIO là module DataForSEO đầu tiên được tạo.
</details>